# Step 2 of 3 — Modelling (cache → tests → training) · Iteration 4

**Reads:** the files written by Step 1 (`local_train_data/`, `shared_calendar.parquet`,
`category_vocab.json`, `column_roles.json`).
**Writes:** `series_cache/` (one `.npz` per series, `spend_paths.npy`, `daily_total_sales.npy` and index tables), the trained model under `darts_logs/`,
and `run_metadata.json`, which the prediction notebook reads.

**What changed vs iteration 3**
1. **New time frames.** Train to 2026-06-14, scored validation 2026-06-15 → 2026-09-30 (17 windows per series), forecast 2026-10-01 → 2026-12-31 (92 days).
2. **Per-series covariates.** Until now every series read the same covariate table. The two spend columns differ by series (model family and state), so each series now gets
   *the shared calendar plus its own two spend columns*. To keep RAM flat, series whose spend paths are identical share **one** object (Section 3 finds them by content, so no state column is needed).
3. **New Test 4** checks those covariates: right columns and dates, binary flags are 0/1, scaled columns are small, the scaling constants really were applied, and different spend paths reach different series.
   **Test 2b** checks the network was built with the expected number of future inputs.
4. **Cache extras:** `daily_total_sales.npy` (all-series sales per day) lets Step 3 rebuild the historical month shapes for the Oct–Dec window.
5. The petrol and spend scaling happened in Step 1. Nothing is scaled in this notebook.

**The embedding fix from iteration 3 (unchanged).** In iteration 2 the static columns (dealer, family, colour, city…)
reached the network as plain numbers. The training log showed 0 embedding parameters. We traced the cause
in the Darts 0.40 source and reproduced it:

> On its first build, `TFTModel` replaces its `categorical_embedding_sizes` dictionary with internal names
> (`static_covariate_0`, …). If the **same model object** is built a second time — after an interrupted or
> failed `fit()`, or after `lr_find()` — the real column names no longer match anything. Darts then silently
> treats every category as a number.

**Defences in this notebook:**
1. **Test 1 (data):** codes are whole numbers inside each embedding table, with column names matching the configuration.
2. **Test 2 (model):** a throwaway model is trained for 2 batches. Then we inspect how the network treats each
   column: every column must be listed as `EMBEDDING`, with a table of the configured size.
3. **Test 3 (behaviour):** a number is multiplied by a weight, so codes 1, 2, 3 land on a straight line.
   Embedding rows are independent vectors, so they don't. We check the rows are *not* on a line.
4. **Negative control:** the same tests on a model built **without** embeddings must fail. This shows the
   tests can actually detect the problem.
5. **Guard on the real run:** `EmbeddingGate` re-checks the real model just before its first training batch
   and stops the run if the check fails. The model is always created in the same cell that trains it.

In [ ]:
import os
import json
import glob
import gc
import shutil
import hashlib
import collections.abc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import darts
from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood
from pytorch_lightning import Callback
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from tft_checks import embedding_report, assert_embeddings, EmbeddingGate

print("darts", darts.__version__, "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## Section 0 — Configuration (dates and roles come from Step 1)
`VAL_START` is the first day whose sales are **scored**. The validation series themselves start 365 days earlier
(`VAL_SERIES_START`) because the encoder needs a year of history before the first scored day.

In [ ]:
BASE = os.getcwd()
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)

time_col, group_col, target_col = ROLES["time_col"], ROLES["group_col"], ROLES["target_col"]
FREQ              = ROLES["freq"]
static_covariates = ROLES["static_covariates"]
binary_covariates = ROLES["binary_covariates"]
shared_covariates = ROLES["shared_covariates"]        # in the shared calendar: same for every series
series_covariates = ROLES["series_covariates"]        # the two spend columns: differ by series
future_covariates = ROLES["future_covariates"]        # what the network sees, in this order
SCALING           = ROLES["scaling"]
assert future_covariates == shared_covariates + series_covariates, \
    "column_roles.json: future_covariates must be shared_covariates followed by series_covariates."

TRAIN_START      = pd.Timestamp(ROLES["train_start"])
TRAIN_END        = pd.Timestamp(ROLES["train_end"])
VAL_START        = pd.Timestamp(ROLES["val_start"])           # first SCORED validation day
VAL_END          = pd.Timestamp(ROLES["val_end"])
VAL_SERIES_START = pd.Timestamp(ROLES["val_series_start"])    # encoder start of the first validation window
FORECAST_START   = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END     = pd.Timestamp(ROLES["forecast_end"])

INPUT_CHUNK_LENGTH  = ROLES["input_chunk_length"]      # 365
OUTPUT_CHUNK_LENGTH = ROLES["output_chunk_length"]     # 92
MIN_LEN = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH      # shortest series that yields one window

assert VAL_START > TRAIN_END, "Scored validation days must come after TRAIN_END."
assert VAL_SERIES_START == VAL_START - pd.Timedelta(days=INPUT_CHUNK_LENGTH), \
    "val_series_start in column_roles.json is not VAL_START minus the input length -- re-run Step 1."
assert FORECAST_START == VAL_END + pd.Timedelta(days=1)

HIST_DATES = pd.date_range(TRAIN_START, VAL_END, freq=FREQ)          # days with sales
FULL_DATES = pd.date_range(TRAIN_START, FORECAST_END, freq=FREQ)     # days with covariates
N_HIST, N_FULL = len(HIST_DATES), len(FULL_DATES)

local_train_dir = os.path.join(BASE, "local_train_data")
CACHE_DIR       = os.path.join(BASE, "series_cache")

# festive sample weighting: weight = 1 + FESTIVE_WEIGHT on any day where a festive-window column is non-zero
FESTIVE_WEIGHT = 3.0
PENALTY_COLS = ([f"N-{i}" for i in range(16, 0, -1)] + ["N"] + [f"N+{i}" for i in range(1, 11)]
                + [f"D-{i}" for i in range(3, 0, -1)] + ["D"] + [f"D+{i}" for i in range(1, 7)]
                + ["C"] + [f"C+{i}" for i in range(1, 7)])
PENALTY_COLS = [c for c in PENALTY_COLS if c in future_covariates]

def safe_name(key):
    return str(key).replace("<>", "_").replace("/", "_").replace("\\", "_")

_val_days = (VAL_END - VAL_START).days + 1
N_VAL_WINDOWS = _val_days - OUTPUT_CHUNK_LENGTH + 1
print(f"TRAIN targets      : {TRAIN_START.date()} -> {TRAIN_END.date()}")
print(f"VAL scored dates   : {VAL_START.date()} -> {VAL_END.date()} ({_val_days} days; {N_VAL_WINDOWS} windows per series)")
print(f"VAL encoder starts : {VAL_SERIES_START.date()} (series length {(VAL_END - VAL_SERIES_START).days + 1} = {INPUT_CHUNK_LENGTH} input + {_val_days} scored)")
print(f"FORECAST           : {FORECAST_START.date()} -> {FORECAST_END.date()} ({OUTPUT_CHUNK_LENGTH} days)")
print(f"future covariates  : {len(future_covariates)} = {len(shared_covariates)} shared + {len(series_covariates)} per-series")
print(f"festive-weighted columns: {len(PENALTY_COLS)}")

## Section 1 — Embedding sizes
Each static column gets a table with one row per category, plus row 0 for "unknown".
The vector size per category follows a common rule of thumb, `1.6 × categories^0.56`, capped at 16.

Why cap dealer and city at 16 (iteration 2 asked for 50): each series' own 365-day history already tells the
model its dealer's sales level. The dealer vector only needs to capture *how dealers differ beyond that*.
A 50-wide vector for ~1,100 dealers is mostly room to memorise noise. Change `EMBEDDING_DIM_OVERRIDES`
to experiment.

In [ ]:
with open(os.path.join(BASE, ROLES["vocab_path"])) as f:
    VOCAB = json.load(f)
assert list(VOCAB) == static_covariates, "Vocabulary columns differ from the static covariates."
assert all(v[0] == "__UNKNOWN__" for v in VOCAB.values())

def default_dim(n_categories):
    return int(min(16, max(2, round(1.6 * n_categories ** 0.56))))

EMBEDDING_DIM_OVERRIDES = {}          # e.g. {"PARENT_DEALER_CODE": 8}

EMBEDDING_SIZES = {
    c: (len(VOCAB[c]), EMBEDDING_DIM_OVERRIDES.get(c, default_dim(len(VOCAB[c]) - 1)))
    for c in static_covariates
}
print(pd.DataFrame([dict(column=c, categories=len(VOCAB[c]) - 1, table_rows=EMBEDDING_SIZES[c][0],
                         vector_size=EMBEDDING_SIZES[c][1]) for c in static_covariates]).to_string(index=False))

## Section 2 — Shared calendar and festive weights
One covariate table holds everything that is the same for every series: the 0/1 flags, the scaled petrol index and the
calendar features (see Step 1, Section 7). Festive sample weights are looked up by date from the same table.
The calendar is national, so a date's weight is identical for every series.

The two spend columns are **not** in this table. They are added per series in Section 6.

In [ ]:
cal = pd.read_parquet(os.path.join(BASE, ROLES["calendar_path"]))
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.sort_values(time_col).reset_index(drop=True)
assert len(cal) == N_FULL and (cal[time_col].to_numpy() == FULL_DATES.to_numpy()).all(), \
    f"Calendar must have exactly one row per day from {TRAIN_START.date()} to {FORECAST_END.date()}."
_absent = [c for c in shared_covariates if c not in cal.columns]
assert not _absent, f"Calendar is missing columns listed in column_roles.json: {_absent}"

SHARED_COV = TimeSeries.from_dataframe(cal, time_col=time_col, value_cols=shared_covariates,
                                       freq=FREQ).astype(np.float32)

_flag = (cal.set_index(time_col)[PENALTY_COLS] != 0).any(axis=1)
WEIGHT_BY_DATE = (1.0 + FESTIVE_WEIGHT * _flag).astype(np.float32)
print(f"Calendar {cal[time_col].min().date()} -> {cal[time_col].max().date()}, "
      f"{len(shared_covariates)} shared covariates; festive-weighted days: {int(_flag.sum())}")
print(f"Festive-weighted days inside the scored validation dates: "
      f"{int(_flag.loc[VAL_START:VAL_END].sum())} of {_val_days}  "
      f"(early stopping watches these days; the festive peak itself is not in them)")
print(f"Festive-weighted days inside the forecast window        : "
      f"{int(_flag.loc[FORECAST_START:FORECAST_END].sum())} of {OUTPUT_CHUNK_LENGTH}")

## Section 3 — Per-series cache
One `.npz` file per series, holding raw daily counts:
- `train_sales` / `train_start` — from the series' first date to `TRAIN_END`
- `val_sales` / `val_start` — from `VAL_SERIES_START` to `VAL_END` (365 input days plus the scored days)

A summary row per series goes to `series_table.parquet`: lengths, totals, the raw category strings and the id of its **spend path**.

**Spend paths.** The two scaled spend columns of a series, from `TRAIN_START` to `FORECAST_END`, form its spend path.
Series that share a model family and a state have identical paths. Each distinct path is stored once in
`spend_paths.npy` (shape: paths × days × 2), and every series just keeps the id of its path. Because paths are matched by
their *values* (rounded to 4 decimals), nothing here needs a state column.

**`REBUILD_CACHE = True`** (the default) deletes the cache and rebuilds it from the chunks, which takes a few
minutes. Set it to `False` only to re-run training on a cache built from the *same* chunks and dates. After you
re-run Step 1 or change any date, it must be `True`.

In [ ]:
REBUILD_CACHE = True

series_table_path = os.path.join(CACHE_DIR, "series_table.parquet")
spend_paths_path  = os.path.join(CACHE_DIR, "spend_paths.npy")
daily_total_path  = os.path.join(CACHE_DIR, "daily_total_sales.npy")

if not REBUILD_CACHE:
    print("Reusing existing cache (REBUILD_CACHE = False).")
    series_table = pd.read_parquet(series_table_path)
    SPEND_PATHS = np.load(spend_paths_path)
else:
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    os.makedirs(CACHE_DIR)
    rows = []
    daily_total = np.zeros(N_HIST, dtype=np.float64)     # all-series NET_SALES per day, TRAIN_START -> VAL_END
    path_of_digest, path_arrays = {}, []
    chunk_files = sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet")))
    assert chunk_files, f"No chunks in {local_train_dir} -- run Step 1 first."
    for ci, path in enumerate(chunk_files):
        df = pd.read_parquet(path)
        df[time_col] = pd.to_datetime(df[time_col])
        for key, g in df.groupby(group_col, sort=False):
            g = g.sort_values(time_col)
            t = g[time_col].reset_index(drop=True)
            assert (t.diff().dropna() == pd.Timedelta(days=1)).all(), f"{key}: dates are not consecutive"

            row = dict(key=str(key), first_date=t.iloc[0], last_date=t.iloc[-1], n_rows=len(t),
                       n_train=0, n_val=0, total_sales=np.nan, spend_path=-1)
            row.update({c: str(g[c].iloc[0]) for c in static_covariates})
            if len(t) != N_FULL or t.iloc[0] != TRAIN_START or t.iloc[-1] != FORECAST_END:
                rows.append(row)          # Section 4 stops with a readable message
                continue

            is_hist = (t <= VAL_END).to_numpy()
            sales = g[target_col].to_numpy(dtype=np.float32)[is_hist]
            assert np.isfinite(sales).all() and (sales >= 0).all() and np.allclose(sales, np.rint(sales)), \
                f"{key}: non-count target"
            t_h = t[is_hist].reset_index(drop=True)
            tr = (t_h <= TRAIN_END).to_numpy()
            va = (t_h >= VAL_SERIES_START).to_numpy()
            payload = dict(val_sales=sales[va],
                           val_start=np.array(str(t_h[va].iloc[0].date())) if va.any() else np.array(""))
            if tr.any():
                payload.update(train_sales=sales[tr], train_start=np.array(str(t_h[tr].iloc[0].date())))
            np.savez(os.path.join(CACHE_DIR, f"{safe_name(key)}.npz"), **payload)
            daily_total += sales

            total = float(sales.sum())
            pid = -1                                    # series with no sales at all are excluded later
            if total > 0:
                spend = g[series_covariates].to_numpy(dtype=np.float32)       # (N_FULL, 2), already scaled in Step 1
                assert np.isfinite(spend).all(), f"{key}: spend columns contain NaN/inf"
                digest = hashlib.md5(np.round(spend, 4).tobytes()).hexdigest()
                pid = path_of_digest.setdefault(digest, len(path_of_digest))
                if pid == len(path_arrays):
                    path_arrays.append(spend)
            row.update(n_train=int(tr.sum()), n_val=int(va.sum()), total_sales=total, spend_path=int(pid))
            rows.append(row)
        del df
        gc.collect()
        print(f"  chunk {ci+1}/{len(chunk_files)} -- series so far: {len(rows):,} | distinct spend paths so far: {len(path_arrays):,}")
    series_table = pd.DataFrame(rows)
    series_table.to_parquet(series_table_path, index=False)
    SPEND_PATHS = (np.stack(path_arrays).astype(np.float32) if path_arrays
                   else np.zeros((0, N_FULL, len(series_covariates)), dtype=np.float32))
    np.save(spend_paths_path, SPEND_PATHS)
    np.save(daily_total_path, daily_total)
print(f"Series in cache: {len(series_table):,} | spend paths: {len(SPEND_PATHS):,}")

## Section 4 — Check every series is complete, then flag which to use
The source table has a row for every series on every date (zero-sales days included). So every series runs
from `TRAIN_START` to `FORECAST_END`, with sales up to `VAL_END`, and is long enough for training, validation and forecasting.
The assertion makes that explicit. If the table ever changes, this cell stops instead of silently
mis-sorting series.

The only series excluded are those with zero sales over the whole history. The `in_train` / `in_val` /
`in_forecast` flags are kept because the sequences below and the prediction notebook read them.

The second half of the cell summarises the spend paths, because they decide how much RAM the covariates need:
- **paths vs. series**: if spend follows model family × state, expect at most (families × about 36) paths;
- **RAM estimate**: paths × days × covariate columns × 4 bytes. Above `MAX_COV_RAM_GB` the covariates are built on demand instead of being kept in RAM (slower, but flat memory);
- **flat forecast spend**: paths whose Oct–Dec values all equal the last history day. A high count means the plan was carried forward, not planned.

In [ ]:
st = series_table
short = st[(st["n_rows"] != N_FULL) | (st["first_date"] != TRAIN_START) | (st["last_date"] != FORECAST_END)]
assert short.empty, (
    f"{len(short):,} series do not span {TRAIN_START.date()} -> {FORECAST_END.date()} ({N_FULL} days), e.g.\n"
    f"{short[['key', 'first_date', 'last_date', 'n_rows']].head().to_string(index=False)}")

st["dead"] = st["total_sales"] == 0
st["in_train"] = st["in_val"] = st["in_forecast"] = ~st["dead"]
print(f"series: {len(st):,} | excluded (no sales at all): {int(st.dead.sum()):,} | used: {int((~st.dead).sum()):,}")

# ---- spend paths -------------------------------------------------------------
live = st[~st["dead"]]
U = len(SPEND_PATHS)
assert U > 0, "No spend paths were found."
assert live["spend_path"].between(0, U - 1).all(), "Some live series have no spend path."
assert SPEND_PATHS.shape == (U, N_FULL, len(series_covariates)), f"spend_paths.npy has shape {SPEND_PATHS.shape}"
assert np.isfinite(SPEND_PATHS).all()

n_family = live["MODEL_FAMILY"].nunique()
per_path = live.groupby("spend_path").size()
fam_per_path = live.groupby("spend_path")["MODEL_FAMILY"].nunique()
print(f"spend paths: {U:,} for {len(live):,} live series and {n_family} model families "
      f"(about {U / max(n_family, 1):.1f} paths per family)")
print(f"series per path: min {per_path.min()}, median {per_path.median():.0f}, max {per_path.max()}")
print(f"paths shared by more than one model family: {int((fam_per_path > 1).sum())}")
if U > 40 * n_family:
    print("*** WARNING: far more paths than model family x state would give (a state count of about 36 per family). "
          "Spend may vary by something finer than state, or two series with the same plan carry tiny numeric differences. "
          "Check before training: the covariates will use more RAM and the 'one path per family x state' assumption is wrong. ***")

MAX_COV_RAM_GB = 6.0
est_gb = U * N_FULL * len(future_covariates) * 4 / 1e9
COV_IN_RAM = est_gb <= MAX_COV_RAM_GB
print(f"covariate RAM if all paths are kept: {est_gb:.2f} GB (limit {MAX_COV_RAM_GB} GB) -> "
      f"{'kept in RAM' if COV_IN_RAM else 'BUILT ON DEMAND (slower)'}")

# ---- is the Oct-Dec plan real or carried forward? ----------------------------
i_last  = (VAL_END - TRAIN_START).days
i_fc0   = (FORECAST_START - TRAIN_START).days
flat = np.isclose(SPEND_PATHS[:, i_fc0:, :], SPEND_PATHS[:, i_last:i_last + 1, :], atol=1e-4).all(axis=(1, 2))
print(f"spend paths whose whole forecast window equals the {VAL_END.date()} value: {int(flat.sum()):,} of {U:,}")
if flat.mean() > 0.5:
    print("*** WARNING: most spend paths are flat across the forecast window -- the planned spend looks carried forward. ***")
_corr = np.corrcoef(SPEND_PATHS[:, :, 0].ravel(), SPEND_PATHS[:, :, 1].ravel())[0, 1]
print(f"correlation between the two scaled spend columns (across all paths and days): {_corr:.3f}")

## Section 5 — Static codes, and Test 1 (data level)
Every series' category strings are mapped to codes with the Step-1 vocabulary. Unseen strings map to 0 ("unknown").

**What Test 1 can and cannot tell you.** At the data level, embedding codes and ordinal codes look the same.
Both are whole numbers. The difference is only in what the network does with them, which Tests 2 and 3
check. Test 1 makes sure the codes are *valid embedding indices*:
- whole numbers, no blanks;
- between 0 and the table size − 1 (a code outside the table would crash an embedding lookup);
- columns in exactly the configured order, because Darts matches tables to columns by name and position.

In [ ]:
CODE_MAPS = {c: {v: i for i, v in enumerate(VOCAB[c])} for c in static_covariates}
codes = pd.DataFrame({c: st[c].map(CODE_MAPS[c]).fillna(0).astype(np.int64) for c in static_covariates})
n_unknown = (codes == 0).sum()
print("series with an unknown (code 0) category:", n_unknown[n_unknown > 0].to_dict() or "none")

series_index = pd.concat([st[["key", "in_train", "in_val", "in_forecast", "spend_path"]], codes], axis=1)
series_index.to_parquet(os.path.join(CACHE_DIR, "series_index.parquet"), index=False)


def test_static_codes(codes_df):
    assert list(codes_df.columns) == list(EMBEDDING_SIZES), \
        f"Column order {list(codes_df.columns)} differs from the embedding configuration."
    for c in codes_df.columns:
        v = codes_df[c]
        assert v.notna().all(), f"{c}: blank codes"
        assert np.array_equal(v, np.round(v)), f"{c}: codes must be whole numbers"
        assert v.min() >= 0 and v.max() < EMBEDDING_SIZES[c][0], \
            f"{c}: codes {v.min()}..{v.max()} fall outside table rows 0..{EMBEDDING_SIZES[c][0] - 1}"
    print(f"TEST 1 PASSED -- {len(codes_df):,} series x {codes_df.shape[1]} columns are valid embedding indices.")


test_static_codes(codes)

## Section 6 — Lazy sequences
Darts receives "lists" of series that are read from disk only when it asks for one. This is how ~32K series fit in RAM.
- `TargetSeq` — daily counts + the series' static codes (one row, one column per static covariate).
- `WeightSeq` — the festive weight for each day, looked up from the calendar by date.
- `SeriesCov` — **new.** The future covariates of each series: the shared calendar columns followed by that series' two spend columns.
  Two series with the same spend path get the *same* `TimeSeries` object, so memory grows with the number of distinct paths, not the number of series.

In [ ]:
class TargetSeq(collections.abc.Sequence):
    def __init__(self, keys, code_rows, split, cache_in_ram=True):
        self.keys, self.code_rows, self.split = list(keys), code_rows, split
        self._ram = {} if cache_in_ram else None

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        if self._ram is not None and i in self._ram:
            return self._ram[i]
        with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
            sales, start = z[f"{self.split}_sales"], str(z[f"{self.split}_start"])
        ts = TimeSeries.from_times_and_values(
            pd.date_range(start, periods=len(sales), freq=FREQ),
            sales.reshape(-1, 1).astype(np.float32),
            columns=[target_col],
            static_covariates=pd.DataFrame([self.code_rows[i]], columns=static_covariates, dtype=np.float32),
        )
        if self._ram is not None:
            self._ram[i] = ts
        return ts


class WeightSeq(collections.abc.Sequence):
    def __init__(self, target_seq):
        self.target_seq = target_seq

    def __len__(self):
        return len(self.target_seq)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        ts = self.target_seq[i]
        w = WEIGHT_BY_DATE.reindex(ts.time_index).to_numpy()
        assert not np.isnan(w).any(), "a series date is missing from the calendar"
        return TimeSeries.from_times_and_values(ts.time_index, w.reshape(-1, 1).astype(np.float32),
                                                columns=["WEIGHT"])


COV_COLUMNS = list(future_covariates)
CAL_INDEX   = SHARED_COV.time_index
CAL_VALUES  = SHARED_COV.values(copy=False)                    # (N_FULL, n shared columns)
assert list(SHARED_COV.components) == shared_covariates
assert len(CAL_INDEX) == N_FULL == SPEND_PATHS.shape[1]


class SeriesCov(collections.abc.Sequence):
    """Future covariates, one TimeSeries per series: the shared calendar columns followed by
    that series' spend columns. Series that share a spend path share one TimeSeries object."""

    def __init__(self, path_ids, cache_in_ram=True):
        self.path_ids = np.asarray(path_ids, dtype=np.int64)
        assert ((self.path_ids >= 0) & (self.path_ids < len(SPEND_PATHS))).all(), "a series has no valid spend path"
        self._ram = {} if cache_in_ram else None

    def _build(self, pid):
        values = np.concatenate([CAL_VALUES, SPEND_PATHS[pid]], axis=1).astype(np.float32, copy=False)
        return TimeSeries.from_times_and_values(CAL_INDEX, values, columns=COV_COLUMNS)

    def __len__(self):
        return len(self.path_ids)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        pid = int(self.path_ids[i])
        if self._ram is None:
            return self._build(pid)
        if pid not in self._ram:
            self._ram[pid] = self._build(pid)
        return self._ram[pid]


def make_sets(mask_col, split, shuffle_seed=None):
    sel = series_index[series_index[mask_col]]
    if shuffle_seed is not None:
        # Darts reads validation windows in series order and never shuffles them. With
        # limit_val_batches=500 (500 x 256 = 128K windows, N_VAL_WINDOWS per series) only the first
        # 128K / N_VAL_WINDOWS series would ever be scored, and they are the first dealers by code.
        # A fixed random order makes that subset representative and identical on every run.
        sel = sel.sample(frac=1.0, random_state=shuffle_seed)
    target = TargetSeq(sel["key"], sel[static_covariates].to_numpy(), split)
    cov = SeriesCov(sel["spend_path"].to_numpy(), cache_in_ram=COV_IN_RAM)
    return target, cov, WeightSeq(target)


train_seq, train_cov, train_w = make_sets("in_train", "train")
val_seq,   val_cov,   val_w   = make_sets("in_val",   "val", shuffle_seed=42)
assert len(train_seq) == len(train_cov) and len(val_seq) == len(val_cov)
print(f"train series: {len(train_seq):,} | val series: {len(val_seq):,}")
print(f"validation windows scored per epoch (limit_val_batches=500, batch 256): "
      f"about {min(500 * 256, len(val_seq) * N_VAL_WINDOWS):,} of {len(val_seq) * N_VAL_WINDOWS:,}")

_probe = train_seq[0]
print("Probe series static covariates (codes, as the model receives them):")
print(_probe.static_covariates)
test_static_codes(_probe.static_covariates.reset_index(drop=True))

## Section 6b — Test 4 (covariates)
Test 1 covers the static codes. Test 4 covers what is new this iteration: the per-series covariates.
- **4a** For a sample of spend paths: the columns are exactly the ones in `column_roles.json` and in the same order, the dates run from `TRAIN_START` to `FORECAST_END`, nothing is NaN, the binary columns hold only 0 and 1, and no continuous column is larger than 8 standard deviations (an unscaled value would be 15 or more).
- **4b** The spend arrays held here really were scaled with the constants in `column_roles.json`: weighted over the training series and dates, the mean must be near 0 and the standard deviation near 1. Petrol is checked the same way on the calendar.
- **4c** Wiring: two different spend paths give identical calendar columns but different spend columns, and the same path always gives the same covariates.

In [ ]:
MAX_ABS_Z = 8.0
_BIN_IDX    = [COV_COLUMNS.index(c) for c in binary_covariates]
_NONBIN_IDX = [i for i in range(len(COV_COLUMNS)) if i not in set(_BIN_IDX)]
N_SHARED = len(shared_covariates)


def test_covariates():
    ids = np.unique(train_cov.path_ids)
    assert len(ids) > 0

    # ---- 4a ----
    for pid in ids[:25]:
        ts = SeriesCov([pid], cache_in_ram=False)[0]
        assert list(ts.components) == COV_COLUMNS, f"path {pid}: covariate columns differ from column_roles.json"
        assert ts.start_time() == TRAIN_START and ts.end_time() == FORECAST_END, \
            f"path {pid}: covariates do not span {TRAIN_START.date()} -> {FORECAST_END.date()}"
        v = ts.values(copy=False)
        assert v.shape == (N_FULL, len(COV_COLUMNS))
        assert np.isfinite(v).all(), f"path {pid}: NaN/inf in covariates"
        assert np.isin(v[:, _BIN_IDX], (0.0, 1.0)).all(), f"path {pid}: a binary covariate is not 0/1"
        assert np.abs(v[:, _NONBIN_IDX]).max() <= MAX_ABS_Z, \
            f"path {pid}: a continuous covariate exceeds {MAX_ABS_Z} -- was it scaled?"
    print(f"TEST 4a PASSED -- {min(25, len(ids))} sampled spend paths have the right columns, dates and value ranges.")

    # ---- 4b ----
    live_ids = series_index.loc[series_index["in_train"], "spend_path"].to_numpy()
    counts = np.bincount(live_ids, minlength=len(SPEND_PATHS)).astype(np.float64)
    n_tr = (TRAIN_END - TRAIN_START).days + 1
    for j, c in enumerate(series_covariates):
        x = SPEND_PATHS[:, :n_tr, j].astype(np.float64)
        mean = (counts * x.mean(axis=1)).sum() / counts.sum()
        var = (counts * (x ** 2).mean(axis=1)).sum() / counts.sum() - mean ** 2
        std = float(np.sqrt(max(var, 0.0)))
        print(f"   {c:30s} training mean = {mean:+.3f}   std = {std:.3f}")
        assert abs(mean) < 0.25 and abs(std - 1.0) < 0.25, (
            f"{c}: the spend arrays are not mean 0 / std 1 over the training series and dates. "
            "Step 1's scaling was not applied, or column_roles.json belongs to a different run.")
    for c in ROLES["shared_continuous_covariates"]:
        z = cal.loc[cal[time_col] <= TRAIN_END, c].to_numpy(dtype=np.float64)
        print(f"   {c:30s} training mean = {z.mean():+.3f}   std = {z.std():.3f}")
        assert abs(z.mean()) < 1e-3 and abs(z.std() - 1.0) < 1e-3, f"{c}: calendar column is not z-scored."
    print("TEST 4b PASSED -- the scaling constants in column_roles.json match the data held here.")

    # ---- 4c ----
    if len(ids) >= 2:
        sc = SeriesCov([ids[0], ids[1], ids[0]], cache_in_ram=False)
        a, b, a2 = (sc[i].values(copy=False) for i in range(3))
        assert np.array_equal(a[:, :N_SHARED], b[:, :N_SHARED]), "calendar columns must be identical for every series"
        assert not np.array_equal(a[:, N_SHARED:], b[:, N_SHARED:]), \
            "two different spend paths produced identical spend columns -- wiring problem"
        assert np.array_equal(a, a2), "the same spend path must always give the same covariates"
        print("TEST 4c PASSED -- different spend paths reach different series; calendar columns are shared.")
    else:
        print("TEST 4c SKIPPED -- only one spend path exists, so every series has the same spend.")


test_covariates()

## Section 7 — Model settings
`build_model()` creates a **new** model every time it is called. The training cell calls it right before
`fit()`. If training is interrupted, re-run the whole training cell so a fresh model is built.
Never call `fit()` again on the old object, and never run `lr_find()` on the object you will train
(that's the iteration-2 failure). To find a learning rate, call `lr_find()` on a separate
`build_model(...)`, then pass the result as `LEARNING_RATE`.

Hyperparameters match iteration 3, so the data and covariates are the main change. The dates and the 92-day horizon also changed.

In [ ]:
LEARNING_RATE = None        # None = Darts default (1e-3)


def build_model(model_name, n_epochs=100, callbacks=(), limit_train_batches=1250, limit_val_batches=500,
                save_checkpoints=True, use_embeddings=True, logger=None):
    trainer_kwargs = {
        "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
        "devices": 1,
        "callbacks": list(callbacks),
        "gradient_clip_val": 0.1,
        "precision": "bf16-mixed" if torch.cuda.is_available() else "32-true",
        "limit_train_batches": limit_train_batches,
        "limit_val_batches": limit_val_batches,
    }
    if logger is not None:           # None = leave Darts' default; the smoke tests rely on that
        trainer_kwargs["logger"] = logger
    return TFTModel(
        input_chunk_length=INPUT_CHUNK_LENGTH,
        output_chunk_length=OUTPUT_CHUNK_LENGTH,
        hidden_size=32,
        lstm_layers=4,
        num_attention_heads=4,
        dropout=0.05,
        batch_size=256,
        n_epochs=n_epochs,
        likelihood=NegativeBinomialLikelihood(),
        loss_fn=None,
        use_reversible_instance_norm=False,
        # a COPY, so nothing Darts does to its dictionary can touch ours
        categorical_embedding_sizes=dict(EMBEDDING_SIZES) if use_embeddings else None,
        add_relative_index=True,
        optimizer_kwargs={"lr": LEARNING_RATE} if LEARNING_RATE else None,
        random_state=42,
        save_checkpoints=save_checkpoints,
        force_reset=True,
        model_name=model_name,
        pl_trainer_kwargs=trainer_kwargs,
    )


FIT_COMMON = dict(dataloader_kwargs={"num_workers": 0, "pin_memory": torch.cuda.is_available()})

## Section 8 — Tests 2 and 3 on a throwaway model, plus a negative control
A small model (64 series, 2 batches) is trained with exactly the same settings. Darts builds the network on
the first `fit()`, so this is the earliest point at which we can look inside it.

- **Test 2** lists, per column, how the network treats it. Expected: `EMBEDDING`, with table rows and vector
  size matching Section 1.
- **Test 2b** (new) counts the future inputs the network was built with. It must equal the number of future covariates in
  `column_roles.json` plus one for the relative index Darts adds (`add_relative_index=True`). This proves the spend columns reached the network.
- **Test 3** looks at the learned table rows for codes 1, 2, 3. A numeric input becomes `weight × code + bias`,
  so codes 1→2→3 move by *the same step*: equal gaps, a straight line. Embedding rows are independent, so
  the gaps differ. If any column's rows lie on a line, the test fails.
- **Negative control:** the same checks on a model built with `categorical_embedding_sizes=None` (the
  iteration-2 situation) must fail. If they passed there too, the tests would prove nothing.

In [ ]:
N_SMOKE = 64
smoke_train = TargetSeq(train_seq.keys[:N_SMOKE], train_seq.code_rows[:N_SMOKE], "train")
smoke_cov   = SeriesCov(train_cov.path_ids[:N_SMOKE], cache_in_ram=COV_IN_RAM)


def smoke_fit(use_embeddings):
    m = build_model("smoke_embedding_test", n_epochs=1, limit_train_batches=2, limit_val_batches=1,
                    save_checkpoints=False, use_embeddings=use_embeddings)
    m.fit(series=smoke_train, future_covariates=smoke_cov,
          sample_weight=WeightSeq(smoke_train), verbose=False, **FIT_COMMON)
    return m


# ---- Test 2 ------------------------------------------------------------------
smoke = smoke_fit(use_embeddings=True)
report, n_emb = assert_embeddings(smoke.model, static_covariates, EMBEDDING_SIZES)
print(report.to_string(index=False))
print(f"\nTEST 2 PASSED -- every static column has its own embedding table ({n_emb:,} parameters).")

# ---- Test 2b: the network was built with the expected number of future inputs ---
try:
    _n_future = len(smoke.model.variables_meta["input"]["future_covariate"])
except Exception as e:                    # a different Darts version may name this differently
    _n_future = None
    print(f"TEST 2b SKIPPED -- could not read the network's future inputs ({type(e).__name__}: {e}).")
if _n_future is not None:
    _expected = len(future_covariates) + 1            # +1: the relative index added by add_relative_index=True
    assert _n_future == _expected, (
        f"The network has {_n_future} future inputs but column_roles.json lists {len(future_covariates)} "
        f"covariates (+1 relative index = {_expected}).")
    print(f"TEST 2b PASSED -- the network has {_n_future} future inputs "
          f"({len(future_covariates)} covariates + 1 relative index).")

# ---- Test 3 ------------------------------------------------------------------
tables = smoke.model.input_embeddings.embeddings
for i, c in enumerate(static_covariates):
    W = tables[f"static_covariate_{i}"].weight.detach().float().cpu().numpy()
    if W.shape[0] < 4:          # needs codes 1, 2, 3 (binary columns only have 1 and 2)
        continue
    step_1_to_2, step_2_to_3 = W[2] - W[1], W[3] - W[2]
    on_a_line = np.allclose(step_1_to_2, step_2_to_3, atol=1e-6)
    assert not on_a_line, f"{c}: codes 1, 2, 3 lie on a straight line -- treated as a number"
print("TEST 3 PASSED -- codes are independent learned vectors, not points on a number line.")

# ---- Negative control ---------------------------------------------------------
control = smoke_fit(use_embeddings=False)
print("\nNegative control (no embeddings), as the network sees it:")
print(embedding_report(control.model, static_covariates, EMBEDDING_SIZES)
      [["column", "treated_as", "table_rows"]].to_string(index=False))
try:
    assert_embeddings(control.model, static_covariates, EMBEDDING_SIZES)
    raise AssertionError("Negative control PASSED the check -- the tests cannot detect the problem.")
except RuntimeError:
    print("NEGATIVE CONTROL OK -- the check correctly fails when categories are fed as numbers.")

del smoke, control
gc.collect()

## Section 8b — TensorBoard logging
View with `tensorboard --logdir tb_logs` (run it from the notebook folder).

Logged per run (all under `tb_logs/<MODEL_NAME>/`):
- `train_loss`, `val_loss` — Lightning/Darts already emit these.
- `lr-*` — learning rate, via `LearningRateMonitor`.
- `diag/grad_norm` — global gradient norm *before* clipping, every `GRAD_LOG_EVERY` steps. Clipping is at 0.1, so a norm
  that sits far above 0.1 means every step is being clipped.
- `emb_norm/<column>` and `emb_hist/<column>` — embedding table size and distribution per epoch. Rows that never move
  from their initial values mean that column is not learning.
- `diag/embedding_parameters` and an hparams entry, so runs can be compared in the HParams tab.

In [ ]:
TB_DIR = os.path.join(BASE, "tb_logs")
GRAD_LOG_EVERY = 50          # optimizer steps


class TBDiagnostics(Callback):
    """Extra TensorBoard scalars/histograms. Never raises: logging must not kill a long training run."""

    def __init__(self, static_cols, every=GRAD_LOG_EVERY):
        self.static_cols, self.every = static_cols, every

    @staticmethod
    def _writer(trainer):
        return getattr(trainer.logger, "experiment", None)

    def on_before_optimizer_step(self, trainer, pl_module, optimizer):
        # runs before gradient clipping, so this is the raw norm
        try:
            w = self._writer(trainer)
            if w is None or trainer.global_step % self.every:
                return
            norms = [p.grad.detach().float().norm(2) for p in pl_module.parameters() if p.grad is not None]
            if norms:
                w.add_scalar("diag/grad_norm", torch.stack(norms).norm(2).item(), trainer.global_step)
        except Exception as e:
            print("TBDiagnostics grad_norm skipped:", e)

    def on_train_epoch_end(self, trainer, pl_module):
        try:
            w = self._writer(trainer)
            if w is None:
                return
            net = getattr(pl_module, "model", pl_module)      # Darts 0.40: the Lightning module IS the network
            tables = net.input_embeddings.embeddings
            for i, c in enumerate(self.static_cols):
                W = tables[f"static_covariate_{i}"].weight.detach().float().cpu()
                w.add_scalar(f"emb_norm/{c}", W.norm(dim=1).mean().item(), trainer.current_epoch)
                w.add_histogram(f"emb_hist/{c}", W.numpy().ravel(), trainer.current_epoch)
            w.flush()
        except Exception as e:
            print("TBDiagnostics embeddings skipped:", e)


def make_tb_logger(model_name):
    return TensorBoardLogger(save_dir=TB_DIR, name=model_name, version="", default_hp_metric=False)

## Section 9 — Training
Model creation and `fit()` are in **one cell on purpose**. Re-running this cell always starts from a fresh model.

`EmbeddingGate` runs the Test-2 check on the real model just before the first training batch and stops the
run if it fails. Early stopping watches `val_loss` on the 2026-06-15 → 2026-09-30 validation windows.
Those days sit just before Navratri, so `val_loss` mostly judges the off-season level, not the festive peak.

There is no "retry without sample weights" fallback any more. A failed first `fit()` followed by a second
`fit()` on the same object is exactly how the embeddings were lost in iteration 2. Darts 0.40 supports
`sample_weight`, so the fallback was never needed.

In [ ]:
MODEL_NAME = f"daily_tft_negbin_emb_{datetime.now().strftime('%Y-%m-%d_%H_%M_%S')}"
print("Model name:", MODEL_NAME)

tb_logger = make_tb_logger(MODEL_NAME)
model = build_model(
    MODEL_NAME,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-4, mode="min"),
               EmbeddingGate(static_covariates, EMBEDDING_SIZES),
               LearningRateMonitor(logging_interval="epoch"),
               TBDiagnostics(static_covariates)],
    logger=tb_logger,
)
tb_logger.log_hyperparams(dict(
    input_chunk_length=INPUT_CHUNK_LENGTH, output_chunk_length=OUTPUT_CHUNK_LENGTH,
    hidden_size=32, lstm_layers=4, num_attention_heads=4, dropout=0.05, batch_size=256,
    festive_weight=FESTIVE_WEIGHT, learning_rate=LEARNING_RATE or 1e-3,
    n_train_series=len(train_seq), n_val_series=len(val_seq),
    n_future_covariates=len(future_covariates), n_spend_paths=len(SPEND_PATHS),
    **{f"emb_dim_{c}": EMBEDDING_SIZES[c][1] for c in static_covariates},
))
model.fit(
    series=train_seq, future_covariates=train_cov, sample_weight=train_w,
    val_series=val_seq, val_future_covariates=val_cov, val_sample_weight=val_w,
    verbose=True, **FIT_COMMON,
)

## Section 10 — Reload the best checkpoint, re-check it, save what prediction needs
The prediction notebook loads the **best** checkpoint by name, so we check that exact object here.
`run_metadata.json` carries the model name, the covariate column order and the scaling constants, so nobody has to type them into the prediction notebook.
The spend paths are already in `series_cache/spend_paths.npy`; `shared_cov.pkl` holds the calendar half of the covariates.

In [ ]:
best = TFTModel.load_from_checkpoint(MODEL_NAME, best=True, map_location="cpu")
report, n_emb = assert_embeddings(best.model, static_covariates, EMBEDDING_SIZES)
print(report.to_string(index=False))
print(f"Best checkpoint uses embeddings ({n_emb:,} parameters).")
tb_logger.experiment.add_scalar("diag/embedding_parameters", int(n_emb), 0)
tb_logger.experiment.flush()
print("TensorBoard: tensorboard --logdir", TB_DIR)

SHARED_COV.to_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
metadata = dict(
    model_name=MODEL_NAME,
    created=datetime.now().isoformat(timespec="seconds"),
    iteration=4,
    dates={k: ROLES[k] for k in ["train_start", "train_end", "val_start", "val_end", "val_series_start",
                                 "forecast_start", "forecast_end"]},
    input_chunk_length=INPUT_CHUNK_LENGTH, output_chunk_length=OUTPUT_CHUNK_LENGTH,
    embedding_sizes={c: list(v) for c, v in EMBEDDING_SIZES.items()},
    embedding_parameters=int(n_emb),
    future_covariate_columns=future_covariates,
    shared_covariate_columns=shared_covariates,
    series_covariate_columns=series_covariates,
    scaling=SCALING,
    n_spend_paths=int(len(SPEND_PATHS)),
    festive_weight=FESTIVE_WEIGHT,
    series=dict(train=int(st.in_train.sum()), val=int(st.in_val.sum()), forecast=int(st.in_forecast.sum())),
    versions=dict(darts=darts.__version__, torch=torch.__version__),
)
with open(os.path.join(BASE, "run_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved shared_cov.pkl and run_metadata.json")